In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("环境配置成功！")

pandas: 3.0.5
numpy: 2.4.6
环境配置成功！


In [2]:
from pathlib import Path

DATA_DIR = Path("../data/raw")

print("数据目录：", DATA_DIR.resolve())
print("\n目录中的文件：")

for file in DATA_DIR.iterdir():
    print(file.name)

数据目录： C:\Users\User\OneDrive\大三\项目训练\O2O_Coupon_Analysis\data\raw

目录中的文件：
offline_test.csv
offline_train.csv
online_train.csv


01 数据读取与数据理解

In [3]:
# 读取线下训练数据
offline_train = pd.read_csv(DATA_DIR / "offline_train.csv")

print("数据读取完成")
print("数据行数：", offline_train.shape[0])
print("数据列数：", offline_train.shape[1])

数据读取完成
数据行数： 1754884
数据列数： 7


In [4]:
# 查看前 5 行数据
offline_train.head()

,User_id,Merchant_id,Coupon_id,Discount_rate,Distance,Date_received,Date
0,1439408,2632,NaN,NaN,0.0,NaN,20160217.0
1,1439408,4663,11002.0,150:20,1.0,20160528.0,NaN
2,1439408,2632,8591.0,20:1,0.0,20160217.0,NaN
3,1439408,2632,1078.0,20:1,0.0,20160319.0,NaN
4,1439408,2632,8591.0,20:1,0.0,20160613.0,NaN


In [5]:
# 查看字段名称
print(offline_train.columns.tolist())

['User_id', 'Merchant_id', 'Coupon_id', 'Discount_rate', 'Distance', 'Date_received', 'Date']


In [6]:
# 查看数据基本信息
offline_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1754884 entries, 0 to 1754883
Data columns (total 7 columns):
 #   Column         Dtype  
---  ------         -----  
 0   User_id        int64  
 1   Merchant_id    int64  
 2   Coupon_id      float64
 3   Discount_rate  str    
 4   Distance       float64
 5   Date_received  float64
 6   Date           float64
dtypes: float64(4), int64(2), str(1)
memory usage: 98.8 MB


2. 数据质量检查

In [7]:
# 统计各字段缺失情况
missing_count = offline_train.isnull().sum()
missing_rate = offline_train.isnull().mean() * 100

missing_summary = pd.DataFrame({
    "缺失数量": missing_count,
    "缺失率(%)": missing_rate.round(2)
})

missing_summary

,缺失数量,缺失率(%)
User_id,0,0.00
Merchant_id,0,0.00
Coupon_id,701602,39.98
Discount_rate,701602,39.98
Distance,106003,6.04
Date_received,701602,39.98
Date,977900,55.72


In [8]:
# 检查重复记录
duplicate_count = offline_train.duplicated().sum()

print("重复记录数量：", duplicate_count)
print("重复记录占比：", round(duplicate_count / len(offline_train) * 100, 2), "%")

重复记录数量： 37893
重复记录占比： 2.16 %


In [9]:
# 查看各字段唯一值数量
offline_train.nunique()

User_id          539438
Merchant_id        8415
Coupon_id          9738
Discount_rate        45
Distance             11
Date_received       167
Date                182
dtype: int64

In [10]:
# 查看优惠方式的典型取值
print("优惠方式数量：", offline_train["Discount_rate"].nunique())
print("\n优惠方式示例：")
print(offline_train["Discount_rate"].value_counts().head(20))

优惠方式数量： 45

优惠方式示例：
Discount_rate
30:5      270712
100:10    182554
200:20    111046
20:5       91013
20:1       51705
50:5       47379
100:30     38196
200:30     29327
300:30     28979
50:10      28452
10:5       25925
0.95       20568
10:1       17842
30:1       17654
150:20     17437
100:20     14297
30:10      12692
50:20       8203
0.9         8085
200:50      5585
Name: count, dtype: int64


In [11]:
# 查看距离字段分布
offline_train["Distance"].value_counts(dropna=False).sort_index()

Distance
0.0     826070
1.0     227221
2.0     118413
3.0      76598
4.0      55085
5.0      41452
6.0      32483
7.0      25681
8.0      21436
9.0      17958
10.0    206484
NaN     106003
Name: count, dtype: int64

3. 优惠券样本构建

In [12]:
# 筛选领取过优惠券的记录
coupon_data = offline_train[
    offline_train["Coupon_id"].notna() &
    offline_train["Date_received"].notna()
].copy()

print("原始数据量：", len(offline_train))
print("领券样本量：", len(coupon_data))
print("领券样本占比：", round(len(coupon_data) / len(offline_train) * 100, 2), "%")

原始数据量： 1754884
领券样本量： 1053282
领券样本占比： 60.02 %


In [13]:
coupon_data.isnull().sum()

User_id               0
Merchant_id           0
Coupon_id             0
Discount_rate         0
Distance         106003
Date_received         0
Date             977900
dtype: int64

In [14]:
# 转换日期字段
coupon_data["Date_received"] = pd.to_datetime(
    coupon_data["Date_received"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

coupon_data["Date"] = pd.to_datetime(
    coupon_data["Date"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

coupon_data[["Date_received", "Date"]].head(10)

,Date_received,Date
1,2016-05-28,NaT
2,2016-02-17,NaT
3,2016-03-19,NaT
4,2016-06-13,NaT
6,2016-05-16,2016-06-13
7,2016-04-29,NaT
8,2016-01-29,NaT
9,2016-05-30,NaT
10,2016-05-19,NaT
13,2016-06-06,NaT


In [15]:
# 计算领券到消费之间的间隔天数
coupon_data["days_to_use"] = (
    coupon_data["Date"] - coupon_data["Date_received"]
).dt.days

In [16]:
# 构造目标变量：
# 15天内使用优惠券 = 1，否则 = 0
coupon_data["label"] = (
    coupon_data["days_to_use"].between(0, 15)
).astype(int)

coupon_data[
    ["User_id", "Coupon_id", "Date_received", "Date", "days_to_use", "label"]
].head(20)

,User_id,Coupon_id,Date_received,Date,days_to_use,label
1,1439408,11002.0,2016-05-28,NaT,NaN,0
2,1439408,8591.0,2016-02-17,NaT,NaN,0
3,1439408,1078.0,2016-03-19,NaT,NaN,0
4,1439408,8591.0,2016-06-13,NaT,NaN,0
6,1439408,8591.0,2016-05-16,2016-06-13,28.0,0
7,1832624,7610.0,2016-04-29,NaT,NaN,0
8,2029232,11951.0,2016-01-29,NaT,NaN,0
9,2029232,1532.0,2016-05-30,NaT,NaN,0
10,2029232,12737.0,2016-05-19,NaT,NaN,0
13,2747744,1097.0,2016-06-06,NaT,NaN,0


In [17]:
label_summary = coupon_data["label"].value_counts().sort_index()
label_rate = coupon_data["label"].value_counts(normalize=True).sort_index() * 100

print("标签数量：")
print(label_summary)

print("\n标签占比：")
print(label_rate.round(2))

标签数量：
label
0    988887
1     64395
Name: count, dtype: int64

标签占比：
label
0    93.89
1     6.11
Name: proportion, dtype: float64


In [18]:
# 检查正样本，验证标签构造是否正确
coupon_data.loc[
    coupon_data["label"] == 1,
    ["User_id", "Coupon_id", "Date_received", "Date", "days_to_use", "label"]
].head(10)

,User_id,Coupon_id,Date_received,Date,days_to_use,label
33,1113008,11166.0,2016-05-15,2016-05-21,6.0,1
38,2881376,7531.0,2016-03-21,2016-03-29,8.0,1
69,114747,2366.0,2016-05-23,2016-06-05,13.0,1
76,114747,111.0,2016-02-07,2016-02-18,11.0,1
77,114747,7751.0,2016-01-27,2016-01-28,1.0,1
84,114747,8088.0,2016-03-24,2016-03-31,7.0,1
143,205174,4627.0,2016-02-06,2016-02-14,8.0,1
192,1380272,12034.0,2016-02-01,2016-02-05,4.0,1
193,1380272,7751.0,2016-02-01,2016-02-06,5.0,1
197,1380272,111.0,2016-02-01,2016-02-05,4.0,1


4. 优惠券特征处理

In [19]:
# 判断优惠券是否属于满减类型
coupon_data["is_manjian"] = (
    coupon_data["Discount_rate"].str.contains(":")
).astype(int)

coupon_data[["Discount_rate", "is_manjian"]].head(10)

,Discount_rate,is_manjian
1,150:20,1
2,20:1,1
3,20:1,1
4,20:1,1
6,20:1,1
7,200:20,1
8,200:20,1
9,30:5,1
10,20:1,1
13,50:10,1


In [20]:
def convert_discount_rate(rate):
    """
    将优惠方式统一转换为折扣率
    例如：
    30:5  -> 0.8333
    100:20 -> 0.8
    0.9 -> 0.9
    """
    rate = str(rate)

    if ":" in rate:
        threshold, reduction = rate.split(":")
        threshold = float(threshold)
        reduction = float(reduction)
        return (threshold - reduction) / threshold

    return float(rate)


coupon_data["discount_rate_value"] = (
    coupon_data["Discount_rate"].apply(convert_discount_rate)
)

coupon_data[
    ["Discount_rate", "is_manjian", "discount_rate_value"]
].head(20)

,Discount_rate,is_manjian,discount_rate_value
1,150:20,1,0.866667
2,20:1,1,0.950000
3,20:1,1,0.950000
4,20:1,1,0.950000
6,20:1,1,0.950000
7,200:20,1,0.900000
8,200:20,1,0.900000
9,30:5,1,0.833333
10,20:1,1,0.950000
13,50:10,1,0.800000


In [21]:
def get_discount_threshold(rate):
    """提取满减门槛，例如 100:20 -> 100"""
    rate = str(rate)

    if ":" in rate:
        return float(rate.split(":")[0])

    return 0.0


def get_discount_reduction(rate):
    """提取满减金额，例如 100:20 -> 20"""
    rate = str(rate)

    if ":" in rate:
        return float(rate.split(":")[1])

    return 0.0


coupon_data["discount_threshold"] = (
    coupon_data["Discount_rate"].apply(get_discount_threshold)
)

coupon_data["discount_reduction"] = (
    coupon_data["Discount_rate"].apply(get_discount_reduction)
)

In [22]:
coupon_data[
    [
        "Discount_rate",
        "is_manjian",
        "discount_rate_value",
        "discount_threshold",
        "discount_reduction"
    ]
].head(20)

,Discount_rate,is_manjian,discount_rate_value,discount_threshold,discount_reduction
1,150:20,1,0.866667,150.0,20.0
2,20:1,1,0.950000,20.0,1.0
3,20:1,1,0.950000,20.0,1.0
4,20:1,1,0.950000,20.0,1.0
6,20:1,1,0.950000,20.0,1.0
7,200:20,1,0.900000,200.0,20.0
8,200:20,1,0.900000,200.0,20.0
9,30:5,1,0.833333,30.0,5.0
10,20:1,1,0.950000,20.0,1.0
13,50:10,1,0.800000,50.0,10.0


In [23]:
# 检查直接折扣类型优惠券
coupon_data.loc[
    coupon_data["is_manjian"] == 0,
    [
        "Discount_rate",
        "is_manjian",
        "discount_rate_value",
        "discount_threshold",
        "discount_reduction"
    ]
].head(10)

,Discount_rate,is_manjian,discount_rate_value,discount_threshold,discount_reduction
148,0.9,0,0.90,0.0,0.0
149,0.9,0,0.90,0.0,0.0
174,0.9,0,0.90,0.0,0.0
217,0.95,0,0.95,0.0,0.0
253,0.95,0,0.95,0.0,0.0
262,0.9,0,0.90,0.0,0.0
448,0.9,0,0.90,0.0,0.0
489,0.9,0,0.90,0.0,0.0
495,0.95,0,0.95,0.0,0.0
496,0.95,0,0.95,0.0,0.0


5. 距离与时间特征处理

In [24]:
# 标记距离是否缺失
coupon_data["distance_missing"] = (
    coupon_data["Distance"].isna()
).astype(int)

print(coupon_data["distance_missing"].value_counts())

distance_missing
0    947279
1    106003
Name: count, dtype: int64


In [25]:
# 将未知距离编码为 -1，保留其独立业务含义
coupon_data["distance_value"] = (
    coupon_data["Distance"].fillna(-1)
)

coupon_data[
    ["Distance", "distance_missing", "distance_value"]
].head(20)

,Distance,distance_missing,distance_value
1,1.0,0,1.0
2,0.0,0,0.0
3,0.0,0,0.0
4,0.0,0,0.0
6,0.0,0,0.0
7,0.0,0,0.0
8,1.0,0,1.0
9,0.0,0,0.0
10,0.0,0,0.0
13,NaN,1,-1.0


In [26]:
# 提取领券时间特征
coupon_data["receive_month"] = coupon_data["Date_received"].dt.month
coupon_data["receive_day"] = coupon_data["Date_received"].dt.day
coupon_data["receive_weekday"] = coupon_data["Date_received"].dt.weekday

# 是否周末：周六、周日 = 1
coupon_data["is_weekend"] = (
    coupon_data["receive_weekday"] >= 5
).astype(int)

In [27]:
coupon_data[
    [
        "Date_received",
        "receive_month",
        "receive_day",
        "receive_weekday",
        "is_weekend"
    ]
].head(20)

,Date_received,receive_month,receive_day,receive_weekday,is_weekend
1,2016-05-28,5,28,5,1
2,2016-02-17,2,17,2,0
3,2016-03-19,3,19,5,1
4,2016-06-13,6,13,0,0
6,2016-05-16,5,16,0,0
7,2016-04-29,4,29,4,0
8,2016-01-29,1,29,4,0
9,2016-05-30,5,30,0,0
10,2016-05-19,5,19,3,0
13,2016-06-06,6,6,0,0


6. 保存基础处理数据

In [28]:
# 保存基础处理后的优惠券数据
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

coupon_data.to_parquet(
    PROCESSED_DIR / "coupon_basic.parquet",
    index=False
)

print("保存完成")
print("数据形状：", coupon_data.shape)
print("保存位置：", (PROCESSED_DIR / "coupon_basic.parquet").resolve())

保存完成
数据形状： (1053282, 19)
保存位置： C:\Users\User\OneDrive\大三\项目训练\O2O_Coupon_Analysis\data\processed\coupon_basic.parquet
